In [1]:
# pip install -U ipywidgets tqdm
# pip install torch transformers datasets
# pip install -U huggingface_hub


In [2]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '684844ebbe03915433e23af5', 'name': 'chadishere', 'fullname': 'Mohammadamin Ahanin', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/a96e8cc64d929630ad26c129bd66f976.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-09T19:26:04.000Z'}}


In [3]:
# import os

# os.environ["HF_HUB_DISABLE_XET"] = "1"

# print("Xet disabled")

In [4]:
from tqdm.auto import tqdm

In [5]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch: 2.14.0+cpu
Transformers: 5.17.0
Datasets: 5.0.1
Device: CPU


In [6]:
from datasets import load_dataset

data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


In [7]:
data["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Downloading model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("Done!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done!


In [9]:
model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [10]:
from transformers import pipeline

pipe = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    device=-1
)

print("Pipeline ready!")

Pipeline ready!


In [11]:
text = data["test"][0]["text"]

print("Review:")
print(text)

print("\nPrediction:")
print(pipe(text))

Review:
lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .

Prediction:
[{'label': 'positive', 'score': 0.9546052813529968}]


In [12]:
# from transformers.pipelines.pt_utils import KeyDataset
# from tqdm.auto import tqdm

# y_pred = []

# for out in tqdm(pipe(KeyDataset(data["test"], "text"))):
#     scores = {x["label"]: x["score"] for x in out}
#     y_pred.append(max(scores, key=scores.get))

  0%|          | 0/1066 [00:00<?, ?it/s]

TypeError: string indices must be integers, not 'str'

In [13]:
test_text = data["test"][0]["text"]

result = pipe(test_text)

print(result)
print(type(result))

[{'label': 'positive', 'score': 0.9546052813529968}]
<class 'list'>


In [15]:
# from transformers.pipelines.pt_utils import KeyDataset
# from tqdm.auto import tqdm

# y_pred = []

# for out in tqdm(pipe(KeyDataset(data["test"], "text"))):
#     y_pred.append(out[0]["label"])

In [16]:
from transformers.pipelines.pt_utils import KeyDataset
from tqdm.auto import tqdm

y_pred = []

for out in tqdm(pipe(KeyDataset(data["test"], "text"))):
    y_pred.append(out["label"])

  0%|          | 0/1066 [00:00<?, ?it/s]

In [17]:
from sklearn.metrics import classification_report

y_true = [
    "positive" if label == 1 else "negative"
    for label in data["test"]["label"]
]

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    negative       0.81      0.69      0.75       533
     neutral       0.00      0.00      0.00         0
    positive       0.91      0.56      0.69       533

    accuracy                           0.63      1066
   macro avg       0.57      0.42      0.48      1066
weighted avg       0.86      0.63      0.72      1066



E:\github repositories\llms\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
E:\github repositories\llms\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
E:\github repositories\llms\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [18]:
from collections import Counter

print(Counter(y_pred))

Counter({'negative': 456, 'positive': 330, 'neutral': 280})


In [19]:
print(model.config.id2label)

{0: 'negative', 1: 'neutral', 2: 'positive'}


In [20]:
print(set(data["test"]["label"]))

{0, 1}


In [21]:
import torch

text = data["test"][0]["text"]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)

print("Text:", text)
print("Probabilities:", probs[0])
print("Labels:", model.config.id2label)

Text: lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .
Probabilities: tensor([0.0052, 0.0402, 0.9546])
Labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [22]:
negative_score = probs[0][0]
positive_score = probs[0][2]

prediction = "positive" if positive_score > negative_score else "negative"

print(prediction)

positive


In [ ]:
import torch
from tqdm.auto import tqdm

y_pred_binary = []

model.eval()

for text in tqdm(data["test"]["text"]):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]

    negative_score = probs[0].item()
    positive_score = probs[2].item()

    prediction = (
        "positive"
        if positive_score > negative_score
        else "negative"
    )

    y_pred_binary.append(prediction)

  0%|          | 0/1066 [00:00<?, ?it/s]

In [ ]:
from collections import Counter

print(Counter(y_pred_binary))